In [28]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy as sp
%matplotlib qt

def bpsk_alt(x,e_vec,nbits,symbol_rate,wincut,fc,Ebit,fs=800e6):
    """
    nbits: number of bits to simulate
    x = the time index length/array 
    tbit = number of samples per symbol
    e_vec = the carrier waveform
    symbol_rate = symbol rate in ksps
    wincut = idk (used in firwin for cutoff)
    fc = carrier frequency
    Ebit = idk (energy per bit?)
    """
    #binary phase shift keyed
    tbit = int( fs / (symbol_rate*1e3) )
    #print('making index range...')
    x = np.arange(nbits*tbit)
    #print('making bit seq...')
    bit_seq = np.random.randint(0,2,size=(int(len(x)/tbit)+1,))
    bit_seq = (2*bit_seq)-1
    pulse = np.ones(tbit)
    #print('making symbol seq...')
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]
    
    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=wincut/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='fft')
    
    #apply carrier signal
    ts = 1/fs
    #print('making carrier signal...')
    e_vec = np.exp(2.j*np.pi*fc*x*ts)
    #e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    #print('modulating...')
    
    sig = sym_seq * e_vec
    return sig,sym_seq,bit_seq


In [29]:

signal = bpsk_alt(400, np.array([]), 2, 5e3, 1.0, 100e6, 1, fs=1000e6)
plt.plot(np.real(signal[0]), label='real')
plt.plot(np.imag(signal[0]), label='imaginary')
plt.title('BPSK Signal')

plt.show()
print(f'symbol sequence: {signal[1]}')
print(f'bit sequence: {signal[2]}')


symbol sequence: [ 5.00000000e-01  5.28638690e-01  5.57218524e-01  5.85680864e-01
  6.13967507e-01  6.42020899e-01  6.69784347e-01  6.97202227e-01
  7.24220189e-01  7.50785355e-01  7.76846506e-01  8.02354272e-01
  8.27261306e-01  8.51522449e-01  8.75094891e-01  8.97938315e-01
  9.20015038e-01  9.41290132e-01  9.61731539e-01  9.81310171e-01
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.00000000e+00  1.00000000e+00  1.00000000e+00  1.00000000e+00
  1.0000

In [30]:
dft = np.fft.fft(signal[0])
plt.plot(np.abs(dft))
plt.show()

In [31]:
def qpsk(x,nbits,symbol_rate,wincut,fc,Ebit,fs=800e6):
    #quad phase shift keyed
    """
    x = the time index length/array 
    nbits: number of bits to simulate
    tbit = number of samples per symbol
    symbol_rate = symbol rate in ksps
    wincut = idk (used in firwin for cutoff)
    fc = carrier frequency
    Ebit = idk (energy per bit?)
    fs = sampling frequency
    """
    #derived number of samples per symbol (this should go inside each rfi generator)
    tbit = int( fs / (symbol_rate*1e3) )
    
    x = np.arange(nbits*tbit)
    bit_seq = np.random.randint(1,5,size=(int(len(x)/tbit)+1,))
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]


    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=1.0/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='auto')


    #apply carrier signal
    ts = 1/fs
    
    #theres a faster way to optimize qpsk i think but this is fine for overnight
    arg = (2.j*np.pi*fc*x*ts) + (1.j*(np.pi/4)*(2*sym_seq-1))
    sig = np.exp(arg)
    #e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    
    #sig = e_vec

    return sig,sym_seq,bit_seq

In [32]:
quadsig = qpsk(400, 2, 2e3, 1.0, 50e6, 1, fs=1000e6)
plt.close('all')
plt.plot(np.real(quadsig[0]), label='real')
plt.plot(np.imag(quadsig[0]), label='imaginary')
plt.title('QPSK Signal')
plt.legend()
plt.show()
print(f'symbol sequence: {quadsig[1]}')
print(f'bit sequence: {quadsig[2]}')


symbol sequence: [2.         2.04583472 2.09165437 2.13744387 2.18318817 2.22887225
 2.27448112 2.31999983 2.3654135  2.4107073  2.45586646 2.50087632
 2.54572227 2.59038984 2.63486462 2.67913235 2.72317887 2.76699016
 2.81055235 2.85385169 2.8968746  2.93960767 2.98203764 3.02415145
 3.0659362  3.10737922 3.148468   3.18919026 3.22953393 3.26948717
 3.30903834 3.34817606 3.38688919 3.42516682 3.4629983  3.50037325
 3.53728154 3.57371332 3.60965899 3.64510927 3.68005514 3.71448787
 3.74839903 3.78178048 3.81462439 3.84692323 3.8786698  3.90985719
 3.94047881 3.97052839 4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         4.         4.         4.
 4.         4.         4.         

In [33]:
QpskDft = np.fft.fft(quadsig[0])
plt.plot(np.abs(QpskDft))
plt.show()

In [34]:
def ask_mod(x,e_vec,nsym,tbit,wincut,symbol_rate,fc,Ebit=0.0,N0=None,fs=800e6, biases= np.array([0.5, 1.0]) ):

    #ASK but with power levels between 0 and 1 to test dc stuff
    tbit = int( fs / (symbol_rate*1e3) )
    x= np.arange(nsym*tbit)
    symsz = int((len(x)/tbit))+1
    bit_seq = np.random.randint(0,len(biases),size=symsz).astype(np.float16)
    for i in range(len(biases)):
        bit_seq[bit_seq==i] = biases[i]
    # print(tbit)
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse)[:len(x)]


    fir_sz = int(0.2*tbit)
    sinc = scipy.signal.firwin(fir_sz, cutoff=wincut/fir_sz, window="rectangular")
    sym_seq = scipy.signal.convolve(sym_seq,sinc,mode='same',method='fft')


    #apply carrier signal
    ts = 1/fs
    e_vec = np.exp(2.j*np.pi*fc*np.arange(len(sym_seq))*ts)
    # e_vec = np.exp(2.j*np.pi * fs/f_sim * x)
    # print(len(sym_seq))
    # print(len(e_vec))
    
    sig = sym_seq * e_vec

    return sig,sym_seq,bit_seq


In [35]:
askSig = ask_mod(400, np.array([]), 10, 5e3, 1.0, 5e3, 100e6, fs=1000e6, biases=np.array([0.2, 0.5, 0.8, 1.0]) )
plt.close('all')
plt.plot(np.real(askSig[0]), label='real')
plt.plot(np.imag(askSig[0]), label='imaginary')
plt.title('ASK Signal')
plt.legend()
plt.show()
print(f'symbol sequence: {askSig[1]}')
print(f'bit sequence: {askSig[2]}')

symbol sequence: [0.09997559 0.10570193 0.1114165  ... 0.11710758 0.1114165  0.10570193]
bit sequence: [0.2 0.8 0.5 0.5 0.2 1.  0.2 0.5 1.  0.2 0.8]


In [36]:
askDft = np.fft.fft(askSig[0])
plt.plot(np.abs(askDft))
plt.show()

In [37]:
def vco_complex(v_in,f0,K0,ts,tbit,nbits):
    """
    Takes an input voltage and returns a variable frequency waveform
    Inputs:
        v_in : input time-indexed voltages (1D)
        f0   : quiescent frequency of oscillator
        K0   : oscillator sensitivity (Hz / V)
        ts   : sampling time (reciprocal of sample rate)
    Output:
        v_out: output time-indexed voltages
    """
    #define stuff
    x = np.tile(np.arange(tbit),nbits)
    phase = np.empty(len(v_in))
    
    #define instantaneous frequencies and phases
    freq = f0 + K0*v_in
    phase = sp.integrate.cumulative_trapezoid(freq,dx=ts,initial=0)
    for i in range(nbits):
        phase[i*tbit:(i+1)*tbit] = phase[i*tbit]
        
    #create waveform
    arg = (2.j*np.pi*x*freq*ts) + 1.j*phase
    v_out = np.exp(arg)
    
    return v_out,freq,phase

#binary freq-shift keying - switch between 2 freqs
def bfsk(nbits,symbol_rate,f0,f1,Ebit=0.0,fs=800e6, N0=None):

    tbit = int( fs / (symbol_rate*1e3) )
    
    #make bit sequence (and turn into seq of +/- 0.5's for VCO)
    bit_seq = np.random.randint(0,high=2,size=nbits)
    pulse = np.ones(tbit)
    sym_seq = np.kron(bit_seq,pulse) - 0.5
    
    #lo-pass filter
    hann = np.hanning(int(tbit*0.2))
    # sym_seq = np.convolve(sym_seq,hann,mode='same')
    # sym_seq = sym_seq/(2*np.max(sym_seq))
    
    ts=1/fs
    Ebit_linear = 10**(Ebit/10.0)

    #define VCO f0,K0 based on inputs
    vco_center = (f0+f1)/2
    #assuming sym_seq = 1 corresponds to voltage = 1V
    vco_sens = (f1-f0)
    
    sig,f,p = vco_complex(sym_seq, vco_center, vco_sens, ts, tbit, nbits)

    sig *= np.sqrt(Ebit_linear)

    return sig,sym_seq,f,p


In [38]:
bfskSig = bfsk(3, 5e3, 100e6, 35e6, fs=1000e6)
plt.close('all')
plt.plot(np.real(bfskSig[0]), label='real')
plt.plot(np.imag(bfskSig[0]), label='imaginary')
plt.title('BFSK Signal')
plt.legend()
plt.show()

print(f'symbol sequence: {bfskSig[1][:20]}')
print(f'frequency from VCO: {bfskSig[2][:20]}')
print(f'phase from VCO: {bfskSig[3][:20]}')


symbol sequence: [0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5
 0.5 0.5]
frequency from VCO: [35000000. 35000000. 35000000. 35000000. 35000000. 35000000. 35000000.
 35000000. 35000000. 35000000. 35000000. 35000000. 35000000. 35000000.
 35000000. 35000000. 35000000. 35000000. 35000000. 35000000.]
phase from VCO: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [39]:
bfskDft = np.fft.fft(bfskSig[0])
plt.plot(np.abs(bfskDft))
plt.show()

In [40]:
def mfrg(Rmethod, nbits, symbol_rate, fc, biases = np.array([0.5,1.0]), f0 = 100e6, f1 = 75e6, Ebit = 1.0, fs = 800e6, nchans = 2048, wincut = 1.0):
    """
    Parameters:
    Rmethod: modulation method to use (e.g. 'ask', 'bpsk', etc.)
    nbits: number of bits to simulate
    symbol_rate: symbol rate in ksps
    fc: center carrier frequency
    biases: for ask_mod, the power levels to use (between 0 and 1)
    f0: for bfsk, the lower frequency
    f1: for bfsk, the higher frequency
    Ebit: energy per bit
    fs: sampling frequency
    nchans: number of channels to simulate (for multi-channel RFI) - centered at fc and seperated by 1 KHz
    wincut: window cut-off frequency

    Returns: a 2D array of shape (nchans, nbits*tbit) containing a simulated RFI signal for each channel using the chosen method
    """
    #derived number of samples per symbol (this should go inside each rfi generator)
    Rmethod = Rmethod.lower().strip()
    tbit = int( fs / (symbol_rate*1e3) )
    # #index time array
    x = np.arange(nbits*tbit)   
    sigarray = np.empty((nchans, len(x)), dtype=np.complex128)
    # print(sigarray.shape)
    
    # pulse = np.ones(tbit)

    if Rmethod == 'ask':
        for i in range(nchans):
            offest = (i-(nchans//2))*1e5
            fc_chan = fc + offest
                
            sig = ask_mod(0, np.array([]), nbits, 0, wincut, symbol_rate, fc_chan, Ebit, None, fs, biases)[0]

            sigarray[i,:] = sig

    elif Rmethod == 'bpsk':
        for i in range(nchans):
            offest = (i-(nchans/2))*1e5
            fc_chan = fc + offest

            sig = bpsk_alt(0, np.array([]), nbits, symbol_rate, wincut, fc_chan, Ebit, fs)[0]

            sigarray[i,:] = sig

    elif Rmethod == 'qpsk':
        for i in range(nchans):
            offest = (i-(nchans//2))*1e5
            fc_chan = fc + offest
            
            sig = qpsk(0, nbits, symbol_rate, wincut, fc_chan, Ebit, fs)[0]

            sigarray[i,:] = sig
    
    #This one is special and requires VCO implementation. 
    elif Rmethod == 'bfsk':
        for i in range(nchans):
            offest = (i-(nchans/2))*1e5
            f1_chan = f1 + offest
            f0_chan = f0 + offest

            sig = bfsk(nbits, symbol_rate, f0_chan, f1_chan, Ebit, fs=fs)[0]

            sigarray[i,:] = sig
    else:
        raise ValueError('Invalid Rmethod. Must be one of: ask, bpsk, qpsk, bfsk')

    return sigarray
     

In [41]:
MultiSig = mfrg('bpsk', 9, 5e3, 150e6, biases= np.array([0.5, 1]), f1=350e6, f0=150e6, wincut=39, fs=1000e6, nchans=2048)
print(MultiSig.shape)
noise = np.random.normal(0, 0.1, size=np.shape(MultiSig))
print(MultiSig.shape)
MultiSig += noise

MultiDFT = np.fft.fft(MultiSig, axis=1)

#pseudo decibel conversion
MDFT_linear = np.abs(MultiDFT)**2
MDFT_db = 10*np.log10(MDFT_linear)

# plt.close('all')
# plt.title('multi frequency qpsk Signal')
# plt.xlabel('temporal frequency')
# plt.ylabel('Frequency ')
# plt.colorbar(label='Magnitude (dB)')
# plt.plot(np.real(MultiSig[:,500]), label='real')
# plt.plot(np.imag(MultiSig[:,2000]), label='imaginary')

# # plt.plot(np.abs(MDFT_db[2000]))
# # plt.plot(np.abs(MDFT_db[200]))

# plt.show()
# plt.imshow(np.abs(MDFT_db))

# print(f'symbol sequence: {MultiSig[1][:20]}')
# print(f'bit sequence: {MultiSig[2][:20]}')
# print(f'phase from VCO: {qpskSig[3][:20]}')

(2048, 1800)
(2048, 1800)


In [42]:
# plt.close('all')
# print(MDFT_db.shape)
# spectra = np.sum(np.abs(MDFT_db), axis=1)
# plt.plot(spectra)
# plt.title('BFSK clean Summed DFT across temporal')
# plt.xlabel('Frequency index')
# plt.ylabel('Magnitude')
# plt.show()
# plt.savefig('plots/BFSK_spectra_clean.png')

In [43]:
#generate all the plots

plt.close('all')
plt.plot(np.real(MultiSig[0]), label='real', linewidth=0.75)
plt.plot(np.imag(MultiSig[0]), label='imaginary', linewidth=0.75)
plt.title('bfsk Signal')
plt.xlabel('Time index')
plt.ylabel('Amplitude')
plt.legend()
plt.show()
# plt.savefig('plots/BFSK_signal_clean.png')

plt.close('all')
plt.plot(np.abs(MDFT_db[2000]), label='channel 2000')
plt.plot(np.abs(MDFT_db[200]), label='channel 200')
plt.title('DFT for two channels')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.legend()
plt.show()
# plt.savefig('plots/BFSK_dft_two_channels_clean.png')

plt.close('all')
plt.imshow(np.abs(MDFT_db))
plt.title('DFT for all channels')
plt.xlabel('Frequency index')
plt.ylabel('Time index')
plt.colorbar(label='Magnitude (dB)' )
plt.show()
# plt.savefig('plots/BFSK_dft_waterfall_clean.png')

plt.close('all')
print(MDFT_db.shape)
spectra = np.sum(np.abs(MDFT_db), axis=0)
plt.plot(spectra, linewidth=0.5)
plt.title('BFSK clean Summed DFT across temporal')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.show()
# plt.savefig('plots/BFSK_spectra_clean.png')


(2048, 1800)


In [44]:
##Spectral Kurtosis##

In [45]:
def SKest (data, N =1 , d =1):
    """
    data: 2d Array (nchans, nsamples)
    N: number of averaged channels (default 1)
    d = shape factor (default 1)
"""
#make sure M is compatible with the temporal length of the data
    bins =8
    M = int(data.shape[1]/bins)
    print(f'M: {M}')

    SKarr = np.empty((data.shape[0], bins))

    for i in range(bins):
        
        S1 = np.sum(np.abs(data[:,(i*M):(i+1)*M])**2, axis=1)
        S2 = np.sum(np.abs(data[:,(i*M):(i+1)*M])**4, axis=1)
        SK = (M*N*d+1)/(M-1) * (M*S2/S1**2 - 1)
        SKarr[:, i] = SK

    #output 2D array of shape (nchans, bins) containing SK values for each bin 
    return SKarr


In [46]:
import scipy.stats as stats

#make pure noise test signal
Noise = np.random.normal(0, 0.1, size=[2048, 3000])

#append Noise signal to RFI signal to test SK esitmators with different duty cycles
SigTest = np.append(MultiSig, Noise, axis=1)


print(SigTest.shape)
#Calc Sk vals
test = SKest(SigTest, N=1, d=1)

print(test) 
plt.close('all')
plt.plot(test, label='SK estimate')
# plt.scatter(range(len(test)), test)
# plt.scatter(range(len(sciTest)), sciTest, label='scipy kurtosis')
plt.xlabel('Channel index')
plt.ylabel('SK Estimate')
plt.show()
# plt.savefig('plots/BPSK_SK_estimate_clean.png')

(2048, 4800)
M: 600
[[0.02370748 0.02681081 0.02987837 ... 2.25400676 1.9517253  1.98228513]
 [0.02139382 0.02450158 0.02634375 ... 2.12179613 2.11351012 2.2510895 ]
 [0.02467864 0.02798752 0.02728943 ... 2.18022053 2.298759   1.80544929]
 ...
 [0.02110315 0.01782276 0.02456454 ... 2.17846361 2.00943176 1.98157411]
 [0.03048629 0.03083571 0.02645513 ... 2.18623392 2.0972331  2.0677572 ]
 [0.02659259 0.02739614 0.0307598  ... 1.6565194  1.79049484 1.71109418]]


In [47]:

# plt.close('all')
RFiMit =SigTest.copy()
print(RFiMit.shape)
# print(np.where(test<2.3))
mask = np.kron(test, np.ones(SigTest.shape[1]//8))
print(mask.shape)
RFiMit[mask < 2.3] = 0
Diag = np.sum(RFiMit, axis=1)
print(np.sum(Diag==0)/2048)
plt.imshow(np.real(RFiMit))
plt.title('Spectral Kurtosis Estimate for BFSK Signal')
plt.xlabel('Channel index')
plt.ylabel('SK Estimate')
plt.show()

(2048, 4800)
(2048, 4800)
0.673828125


In [48]:
np.array

<function numpy.array(object, dtype=None, *, copy=True, order='K', subok=False, ndmin=0, ndmax=0, like=None)>